# M-Pajak: IndoBERT + BERTopic
**Sebelum mulai:** Runtime > Change runtime type > **T4 GPU**. Jalankan sel dari atas ke bawah, satu per satu.
File yang perlu diunggah (Langkah 2): `dataset_berlabel_1000.csv` dan `korpus_mpajak.csv` (dibuat oleh code/02_siapkan_korpus.py).

## 1. Pasang pustaka

In [ ]:
!pip install -q bertopic sentence-transformers transformers accelerate

## 2. Unggah file (jika terputus, ulangi dari sini; atau pasang Google Drive)

In [ ]:
import os
from google.colab import files
if not (os.path.exists('dataset_berlabel_1000.csv') and os.path.exists('korpus_mpajak.csv')):
    files.upload()
print(os.listdir('.'))

## 3. Muat data dan cek GPU

In [ ]:
import re, random, numpy as np, pandas as pd, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)
assert dev == 'cuda', 'Aktifkan GPU: Runtime > Change runtime type > T4 GPU'

def bersih(t):
    t = str(t).lower()
    t = re.sub(r'http\S+', ' ', t)
    t = re.sub(r'(.)\1{3,}', r'\1\1\1', t)
    return re.sub(r'\s+', ' ', t).strip()

d = pd.read_csv('dataset_berlabel_1000.csv')
d = d[d['dipakai']].reset_index(drop=True)      # 991 ulasan berlabel
d['x'] = d['ulasan'].map(bersih)
korpus = pd.read_csv('korpus_mpajak.csv', parse_dates=['at'])
korpus['x'] = korpus['content'].map(bersih)
LAB = ['negatif', 'netral', 'positif']
l2i = {l: i for i, l in enumerate(LAB)}
y = d['sentimen_final'].map(l2i).values
print(len(d), len(korpus))

## 4. Fungsi fine-tuning IndoBERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
MODEL = 'indobenchmark/indobert-base-p1'
tok = AutoTokenizer.from_pretrained(MODEL)
EPOCH, LR, BS, MAXLEN = 4, 2e-5, 16, 128

def enc(texts):
    return tok(list(texts), truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')

def latih(x_tr, y_tr):
    torch.manual_seed(SEED)
    m = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3).to(dev)
    cnt = np.bincount(y_tr, minlength=3)
    w = torch.tensor(len(y_tr) / (3 * np.maximum(cnt, 1)), dtype=torch.float).to(dev)
    lossf = torch.nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    idx = np.arange(len(x_tr))
    steps = EPOCH * int(np.ceil(len(idx) / BS))
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    m.train()
    for ep in range(EPOCH):
        np.random.shuffle(idx)
        for i in range(0, len(idx), BS):
            b = idx[i:i + BS]
            e = {k: v.to(dev) for k, v in enc(x_tr[b]).items()}
            out = m(**e)
            loss = lossf(out.logits, torch.tensor(y_tr[b]).to(dev))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
    return m

@torch.no_grad()
def prediksi(m, texts, bs=64):
    m.eval(); out = []
    texts = np.array(texts)
    for i in range(0, len(texts), bs):
        e = {k: v.to(dev) for k, v in enc(texts[i:i + bs]).items()}
        out.append(torch.softmax(m(**e).logits, -1).cpu().numpy())
    return np.vstack(out)

## 5. Evaluasi 5-fold (lipatan sama persis dengan SVM)
Sekitar 5-10 menit di GPU T4.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, confusion_matrix
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(d), dtype=int)
xs = d['x'].values
for k, (tr, te) in enumerate(skf.split(xs, y), 1):
    m = latih(xs[tr], y[tr])
    oof[te] = prediksi(m, xs[te]).argmax(1)
    print('lipatan', k, 'macro-F1 =', round(f1_score(y[te], oof[te], average='macro'), 3))
    del m; torch.cuda.empty_cache()

print('\n=== IndoBERT, semua 991 ===')
print(classification_report(y, oof, target_names=LAB, digits=3))
ac = (d['sumber'] == 'acak').values
print('=== subset sampel acak ===')
print(classification_report(y[ac], oof[ac], labels=[0, 1, 2], target_names=LAB, digits=3))
print(pd.DataFrame(confusion_matrix(y, oof), index=LAB, columns=LAB))
d['pred_indobert_oof'] = [LAB[i] for i in oof]

## 6. Model akhir (semua 991 ulasan) dan prediksi seluruh korpus

In [ ]:
final = latih(xs, y)
P = prediksi(final, korpus['x'].values)
korpus['pred_indobert'] = [LAB[i] for i in P.argmax(1)]
korpus['p_max'] = P.max(1)
print(korpus['pred_indobert'].value_counts())

## 7. Tren per fase dan per bulan

In [ ]:
import scipy.stats as st
tab = pd.crosstab(korpus['fase'], korpus['pred_indobert'])
print(tab); print((tab.div(tab.sum(1), axis=0) * 100).round(1))
chi2, p, dof, _ = st.chi2_contingency(tab)
print(f'chi-kuadrat={chi2:.1f}, dof={dof}, p={p:.2e}')
korpus['bulan'] = korpus['at'].dt.to_period('M').astype(str)
bln = pd.crosstab(korpus['bulan'], korpus['pred_indobert'])
bln['n'] = bln.sum(1)
bln['pct_negatif'] = (bln['negatif'] / bln['n'] * 100).round(1)
print(bln.to_string())

## 8. BERTopic pada ulasan negatif
Hanya ulasan yang diprediksi **negatif** dan minimal 4 kata. Topik dikurangi menjadi sekitar 20.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

SW = '''yang dan di ke dari untuk ini itu dengan pada saya aku kami kita nya aja saja juga sudah udah belum tidak gak ga nggak tak
bisa dapat akan ada adalah atau karena kalau kalo jadi jd lah deh dong sih kok pun lagi masih mau harus sangat banget bgt sekali lebih
apa apakah bagaimana mengapa kenapa saat waktu sama dgn dg yg tp tapi tetapi namun hanya cuma udh sdh blm tdk dr utk krn jg
aplikasi app apk pajak coretax djp mpajak pak bu min admin tolong mohon terima kasih makasih thanks'''.split()

neg = korpus[(korpus['pred_indobert'] == 'negatif') & (korpus['x'].str.split().str.len() >= 4)].reset_index(drop=True)
print('ulasan negatif untuk BERTopic:', len(neg))
emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2').encode(neg['x'].tolist(), show_progress_bar=True)
topic = BERTopic(
    embedding_model=None,
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=20, metric='euclidean', cluster_selection_method='eom', prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=SW, ngram_range=(1, 2), min_df=3),
    nr_topics=20, calculate_probabilities=False, verbose=True)
neg['topik'], _ = topic.fit_transform(neg['x'].tolist(), emb)
print(topic.get_topic_info()[['Topic', 'Count', 'Name']].to_string())

## 9. Topik: kata kunci, contoh ulasan, dan porsi per fase

In [ ]:
ringkas = []
for t in sorted(neg['topik'].unique()):
    kata = ', '.join(w for w, _ in topic.get_topic(t)[:10]) if t != -1 else '(tidak terklasifikasi)'
    contoh = neg[neg['topik'] == t].sort_values('p_max', ascending=False)['content'].head(5).tolist()
    ringkas.append({'topik': t, 'jumlah': int((neg['topik'] == t).sum()), 'kata_kunci': kata,
                    'contoh_1': contoh[0] if contoh else '', 'contoh_2': contoh[1] if len(contoh) > 1 else '',
                    'contoh_3': contoh[2] if len(contoh) > 2 else ''})
ringkas = pd.DataFrame(ringkas)
pd.set_option('display.max_colwidth', 160)
print(ringkas[['topik', 'jumlah', 'kata_kunci']].to_string())

per_fase = pd.crosstab(neg['topik'], neg['fase'])
per_fase_pct = (per_fase.div(per_fase.sum(0), axis=1) * 100).round(1)
per_fase_pct.columns = [f'fase{c}_%' for c in per_fase_pct.columns]
print(per_fase_pct.to_string())

## 10. Simpan semua hasil dan unduh SATU file zip

In [ ]:
import shutil, os
os.makedirs('hasil', exist_ok=True)
korpus.drop(columns=['x']).to_csv('hasil/korpus_hasil.csv', index=False, encoding='utf-8-sig')
d.drop(columns=['x']).to_csv('hasil/berlabel_oof.csv', index=False, encoding='utf-8-sig')
neg.drop(columns=['x']).to_csv('hasil/negatif_topik.csv', index=False, encoding='utf-8-sig')
ringkas.to_csv('hasil/topik_ringkasan.csv', index=False, encoding='utf-8-sig')
per_fase.to_csv('hasil/topik_per_fase_jumlah.csv', encoding='utf-8-sig')
per_fase_pct.to_csv('hasil/topik_per_fase_persen.csv', encoding='utf-8-sig')
tab.to_csv('hasil/sentimen_per_fase.csv', encoding='utf-8-sig')
bln.to_csv('hasil/sentimen_per_bulan.csv', encoding='utf-8-sig')
shutil.make_archive('hasil_mpajak', 'zip', 'hasil')
from google.colab import files
files.download('hasil_mpajak.zip')